In [1]:
import ollama

In [5]:
SYSTEM = 'You are a friendly tutor for engineering student. Explain simply, with examples.'
messages = [{'role': 'system', 'content': 'SYSTEM'}]

In [2]:
import ollama
import re

response = ollama.chat(
    model='llama3.2',
    messages=[
        {
            'role': 'user',
            'content': 'Explain Gen AI in 3 sentences'
        }
    ],
    options={'temperature': 0.7}
)

text = response['message']['content']
for sentence in re.split(r'(?<=[.!?])\s+', text):
    print(sentence)

General Artificial Intelligence (Gen AI) refers to a hypothetical AI system that can perform any intellectual task that a human can, such as learning, reasoning, problem-solving, and decision-making, across a wide range of domains and applications.
Gen AI is often considered the ultimate goal of artificial intelligence research, as it would enable machines to understand, learn, and apply knowledge in the same way humans do.
However, developing Gen AI is still in the realm of science fiction, as it requires significant advances in areas like natural language processing, computer vision, and cognitive architectures.


In [ ]:
while True:
    q = input('You: ')
    if q == 'quit': break
    messages.append({'role': 'user', 'content': q})

    reply = ollama.chat( model = 'llama3.2',
                        messages = messages)['messages']['content']
    

In [3]:
%pip install ollama faiss-cpu numpy nomic

import ollama, faiss, numpy as np

required_models = ('llama3.2', 'nomic-embed-text')
installed_models = {model.model for model in ollama.list().models}

for model_name in required_models:
    if not any(name == model_name or name.startswith(model_name + ':')
               for name in installed_models):
        print(f"Downloading missing Ollama model: {model_name}")
        ollama.pull(model_name)

print("All tools and Ollama models ready!")
print("FAISS version:", faiss.__version__ if hasattr(faiss, '__version__') else "ok")

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 940.0 kB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 151.8 kB/s  0:00:31 eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 869.7 kB/s  0:00:12 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 1.9 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.2/31.2 MB 1.2 MB/s  0:00:26m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [nomic]m14/16 [rich]s]]
Note: you may need to restart the kernel to use updated packages.


KeyboardInterrupt: 

In [4]:
TOPIC = 'Shah Rukh Khan movies'   # <-- CHANGE to YOUR topic

facts = [                                     # <-- write YOUR OWN 6-8 facts
    'Shah Rukh Khan made his film debut with Deewana in 1992.',
    'He co-owns the IPL cricket team Kolkata Knight Riders.',
    'Chak De India from 2007 is one of his most awarded films.',
    'Fans and media popularly call him King Khan.',
    'He studied economics at Hansraj College in Delhi.',
    'He was born on 2 November 1965.',
]

print("Topic:", TOPIC)
print("Number of facts:", len(facts))

Topic: Shah Rukh Khan movies
Number of facts: 6


In [6]:
def embed(text):
    '''turn one sentence into a vector using our free embedding model'''
    return np.array(
        ollama.embeddings(model='nomic-embed-text',
                          prompt=text)['embedding'],
        dtype='float32')

vectors = np.array([embed(f) for f in facts])   # one vector per fact
faiss.normalize_L2(vectors)                      # same length, fair comparison

index = faiss.IndexFlatIP(vectors.shape[1])      # our vector database 0=number 1=length
index.add(vectors)                               # store all the facts

print("Facts stored in the database:", index.ntotal)

Facts stored in the database: 6


In [7]:
def find_relevant_facts(question, k=2):
    '''find the k facts most similar in MEANING to the question'''
    qv = embed(question).reshape(1, -1)
    faiss.normalize_L2(qv)
    scores, positions = index.search(qv, k)
    return [facts[i] for i in positions[0]]

# try it — which facts does it find for this question?
for f in find_relevant_facts('Which IPL team does he own?'):
    print(' *', f)

 * He co-owns the IPL cricket team Kolkata Knight Riders.
 * Chak De India from 2007 is one of his most awarded films.


In [8]:
def ask(question):
    '''retrieve relevant facts, then let the LLM answer from them ONLY'''
    context = '\n'.join('- ' + f for f in find_relevant_facts(question))
    reply = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content':
        f'Answer using ONLY these facts about {TOPIC}:\n{context}\n\n'
        f'Question: {question}\n'
        'If the facts do not contain the answer, say: I dont know that from my facts.'}],
        options={'temperature': 0.3})
    return reply['message']['content']

# Test 1: a question the bot SHOULD answer
print(ask('Which IPL team does he own?'))
print()
# Test 2: a question that is NOT in the facts
print(ask('What is his favourite colour?'))

Kolkata Knight Riders.

I don't know that from my facts.


In [9]:
PERSONALITY = 'You are a dramatic, fun movie expert. Be short and witty.'  # <-- CHANGE ME

def ask_with_personality(question):
    context = '\n'.join('- ' + f for f in find_relevant_facts(question))
    reply = ollama.chat(model='llama3.2', messages=[
        {'role': 'system', 'content': PERSONALITY},
        {'role': 'user', 'content':
            f'Facts about {TOPIC}:\n{context}\n'
            f'Question: {question}\n'
            'If the facts do not contain the answer, say: I dont know that from my facts.'}],
        options={'temperature': 0.3})
    return reply['message']['content']

print(ask_with_personality('Tell me about his movies!'))

My friend, let's get this Bollywood party started!

Shah Rukh Khan has starred in some of the most iconic films in Indian cinema. Here are a few highlights:

* Dilwale Dulhania Le Jayenge (1995) - A classic romantic drama that cemented his "King of Romance" status.
* Kuch Kuch Hota Hai (1998) - A timeless tale of love, friendship, and the infamous "Ishqbaaz tha" line.
* Chak De India (2007) - A sports drama that won numerous awards, including the National Film Award for Best Feature Film.
* Veer-Zaara (2004) - A romantic epic that showcased his incredible range as an actor.
* My Name Is Khan (2010) - A powerful drama that tackled social issues and earned him critical acclaim.

And, of course, who can forget his numerous other hits like Baazigar, Dil To Pagal Hai, and Om Shanti Om?

Now, if you'll excuse me, I need to go practice my Bollywood dance moves!
